In [0]:
print("Hello World")

- **freight-price**: Price charged by the transporter to transport the product
- **product_weight_g**: Product weight in grams
- **product_score**: Average Product rating
- **customers**: No.of customers in that category
- **weekdays**: No.of weekdays in that month
- **weekends**: No.of weekends in that month
- **holidays**: No.of holidays in that month
- **lag_price**: Previous month price of the product

- Change column names of `product_name_lenght`, `product_description_lenght`
- Better FeatureEng for **weekday, weekend, holiday**

In [0]:
ft_path = "mlcore_dev.retail_prices_1303_v1.transformed_retail_prices_1303_v1_ft"
ft_df = spark.sql(f"SELECT * FROM {ft_path} LIMIT 676")
ft_df.display()

In [0]:
df = ft_df.toPandas()
df

In [0]:
# A New Column: Avg unit_price, which will contain the mean of unit_price for each group of same month, product_category_name.
df["avg_unit_price_by_product"] = df.groupby(["month", "product_category_name"])["unit_price"].transform("mean")

# A New Column: Avg customers, which will contain the mean number of customers for each group of same month, product_category_name.
df["avg_customers_by_product"] = df.groupby(["month", "product_category_name"])["customers"].transform("mean")

# A New Column: Average Competitor Prices, which will contain the mean of 3 competitor prices for each group of same month, product_category_name.
df["avg_comp_price_by_product"] = (
    df[["comp_1", "comp_2", "comp_3"]].mean(axis=1)
)

df["avg_comp_price_by_product"] = df.groupby(["month", "product_category_name"])["avg_comp_price_by_product"].transform("mean")

round_cols = ["freight_price", "unit_price", "s", "comp_1", "comp_2", "comp_3", "lag_price", "avg_comp_price_by_product", "avg_customers_by_product", "avg_unit_price_by_product"]

# Roudning off float value columns to 2 digits.
for col in round_cols:
    df[col] = df[col].round(2)

df

In [0]:
ft_df["lag_units_sold"] = ft_df.groupby("product_category_name")["units_sold"].shift(1)
ft_df

In [0]:
ft_df.count()

In [0]:
gt_path = "mlcore_dev.retail_prices_1303_v1.transformed_retail_prices_1303_v1_gt"
gt_df = spark.sql(f"SELECT * FROM {gt_path} LIMIT 676")
gt_df.display()

In [0]:
ft_df = ft_df.toPandas()
# gt_df = gt_df.toPandas()
ft_df.columns

In [0]:
# Change back to normal year by adding 2017
ft_df["year"] = ft_df["year"] + 2017
# ft_df

In [0]:
# # Add time index
ft_df["time_idx"] = ft_df["year"] * 12 + ft_df["month"]
ft_df["time_idx"] -= ft_df["time_idx"].min()
ft_df.display()

In [0]:
ft_df["time_idx"].unique()

In [0]:
ft_df.columns

1. Groupby `product_category_name`, `time_idx` -> Avg unit_price, Avg of 3 competitors prices, Avg customers